In [36]:
import pandas as pd 
import tensorflow as tf
import sklearn
from sklearn import preprocessing 
from collections import deque
import numpy as np 
import random

#### Defining some parameters that we need to preprocess the data

In [37]:
def classify(current, future): 
    if float(future) > float(current):
        return 1 
    else: 
        return 0 
    
SEQ_LEN= 60
RATIO_TO_PREDICT= "LTC-USD"
FUTURE_RATIO_PREDICT= 3


### Joining four separate dataframes and extracting the columns we need for predicting the future price. 

In [38]:

ratios= ["BCH-USD","BTC-USD","ETH-USD","LTC-USD"]
main_df= pd.DataFrame()

for ratio in ratios: 
    dataset= f"datasets/{ratio}.csv"
    
    df= pd.read_csv(dataset, names= ["time", "low", "high", "open", "close", "volume"])
    df.rename(columns= {"close": f"{ratio}_close", "volume": f"{ratio}_volume"}, inplace= True)
    
    df.set_index("time", inplace= True)
    df= df[[f"{ratio}_close", f"{ratio}_volume"]]
    
    if len(main_df) == 0: 
        main_df= df 
    else: 
        main_df= main_df.join(df)

#Creating a new column future by using the values from third row for all rows. 
main_df["future"]= main_df[f"{RATIO_TO_PREDICT}_close"].shift(-FUTURE_RATIO_PREDICT)

main_df["target"]= list(map(classify,main_df[f"{RATIO_TO_PREDICT}_close"], main_df["future"]))

print(main_df[[f"{RATIO_TO_PREDICT}_close", "future", "target"]].head())

            LTC-USD_close     future  target
time                                        
1528968660      96.580002  96.500000       0
1528968720      96.660004  96.389999       0
1528968780      96.570000  96.519997       0
1528968840      96.500000  96.440002       0
1528968900      96.389999  96.470001       1


In [39]:
time= sorted(main_df.index.values)

#Defining the threshold of last 5% time 
last_five_percent= time[-int(0.05*len(time))]
 
#Defining a validation set as last 5% of the dataset. 
validation_main_df= main_df[(main_df.index >= last_five_percent)]

training_main_df= main_df[(main_df.index < last_five_percent)]

## Defining a function for preprocessing the data 
#### The preprocessing of the data includes scaling the data and turning it into a sequence 

In [40]:
def preprocess_df(df): 
    
    df= df.drop('future', axis= 1)
    
    for col in df.columns: 
        if col != "target":
            df[col]= df[col].pct_change() #Normalizing the data 
            df.dropna(inplace= True)
            df[col]= sklearn.preprocessing.scale(df[col].values)
    df.dropna(inplace= True)
    
    sequential_data= []
    prev_days= deque(maxlen= SEQ_LEN)
    
    for i in df.values: 
        prev_days.append([n for n in i[:-1]])
        if len(prev_days) == SEQ_LEN: 
            sequential_data.append([np.array(prev_days), i[-1]])
            
    np.random.shuffle(sequential_data)
    
    #Lets balance the data 
    buy= []
    dont_buy= []
    
    for seq,tar in enumerate(sequential_data): 
        if tar == 0: 
            dont_buy.append([seq,tar])
        elif tar == 1: 
            buy.append([seq, tar])
            
    lower= min(len(buy), len(dont_buy))
    buy= buy[:lower]
    dont_buy= dont_buy[:lower]
    
    sequential_data= buy + dont_buy
    
    #Lets split the dataset into X (seq) and y (target)
    X= []
    Y= []
    
    for seq, tar in sequential_data: 
        X.append(seq)
        Y.append(tar)
        
    return np.array(X), Y
    



In [41]:
x_train, y_train= preprocess_df(training_main_df)
x_dev, y_dev= preprocess_df(validation_main_df)

print(f"lenth of training data {len(x_train)}, and length of validation data {len(x_dev)}")
print(f"Num of buys {y_train.count(1)}, num of dont_buy {y_train.count(0)} in training data")
print(f"Num of buys {y_dev.count(1)}, num of dont_buy {y_dev.count(0)} in validation data")

lenth of training data 0, and length of validation data 0
Num of buys 0, num of dont_buy 0 in training data
Num of buys 0, num of dont_buy 0 in validation data
